In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!git clone --branch feat/james-ml-pipeline \
    https://github.com/YiMConsulting/YiM-N-ATLAS-Language-Foundry.git \
    /kaggle/working/natlas-foundry

Cloning into '/kaggle/working/natlas-foundry'...
remote: Enumerating objects: 323, done.
remote: Counting objects: 100% (323/323), done.
remote: Compressing objects: 100% (232/232), done.
remote: Total 323 (delta 138), reused 261 (delta 76), pack-reused 0 (from 0)
Receiving objects: 100% (323/323), 186.51 KiB | 2.33 MiB/s, done.
Resolving deltas: 100% (138/138), done.


In [4]:
%cd /kaggle/working/natlas-foundry

!git fetch origin
!git checkout feat/james-ml-pipeline
!git pull --ff-only origin feat/james-ml-pipeline
!git rev-parse HEAD

/kaggle/working/natlas-foundry
Already on 'feat/james-ml-pipeline'
Your branch is up to date with 'origin/feat/james-ml-pipeline'.
From https://github.com/YiMConsulting/YiM-N-ATLAS-Language-Foundry
 * branch            feat/james-ml-pipeline -> FETCH_HEAD
Already up to date.
5c83db80d2356f6e26d02a74bce63d43daca762e


In [9]:
# Natlas collator focused test
!python -m pytest tests/test_natlas_collator.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /kaggle/working/natlas-foundry
configfile: pytest.ini
plugins: typeguard-4.6.0, anyio-4.14.2, langsmith-0.12.1
collected 6 items                                                              

tests/test_natlas_collator.py::test_collator_pads_to_longest_example PASSED [ 16%]
tests/test_natlas_collator.py::test_collator_preserves_prompt_masking PASSED [ 33%]
tests/test_natlas_collator.py::test_collator_rejects_empty_batch PASSED  [ 50%]
tests/test_natlas_collator.py::test_collator_rejects_inconsistent_lengths PASSED [ 66%]
tests/test_natlas_collator.py::test_collator_rejects_missing_padding_token PASSED [ 83%]
tests/test_natlas_collator.py::test_collator_supports_left_padding PASSED [100%]

============================== 6 passed in 8.99s ===============================


In [10]:
# preprocessing and collator test together...
!python -m pytest \
    tests/test_natlas_preprocessing.py \
    tests/test_natlas_collator.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /kaggle/working/natlas-foundry
configfile: pytest.ini
plugins: typeguard-4.6.0, anyio-4.14.2, langsmith-0.12.1
collected 15 items                                                             

tests/test_natlas_preprocessing.py::test_prompt_tokens_are_masked_with_ignore_index PASSED [  6%]
tests/test_natlas_preprocessing.py::test_assistant_and_eot_tokens_remain_supervised PASSED [ 13%]
tests/test_natlas_preprocessing.py::test_input_ids_attention_mask_and_labels_have_equal_lengths PASSED [ 20%]
tests/test_natlas_preprocessing.py::test_empty_input_and_empty_target_are_rejected PASSED [ 26%]
tests/test_natlas_preprocessing.py::test_example_longer_than_max_length_is_rejected PASSED [ 33%]
tests/test_natlas_preprocessing.py::test_mismatched_prompt_boundary_is_rejected PASSED [ 40%]
tests/test_natlas_

In [5]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")
login(secret_value_0)

print("Hugging face Authentication Successful")


Hugging face Authentication Successful


In [13]:
# INSPECTING N_ATLAS PADDING CONFIGUARATION...
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("NCAIR1/N-ATLaS")

print("Tokenizer class:", type(tokenizer).__name__)
print("Padding token:", repr(tokenizer.pad_token))
print("Padding token ID:", tokenizer.pad_token_id)
print("EOS token:", repr(tokenizer.eos_token))
print("EOS token ID:", tokenizer.eos_token_id)
print("Padding side:", tokenizer.padding_side)

config.json:   0%|          | 0.00/875 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/325 [00:00<?, ?B/s]

Tokenizer class: TokenizersBackend
Padding token: '<|eot_id|>'
Padding token ID: 128009
EOS token: '<|eot_id|>'
EOS token ID: 128009
Padding side: right


In [ ]:
# To Validate actual LoRA attachment in Kaggle

In [8]:
# Load model in FP16
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "NCAIR1/N-ATLaS"

assert torch.cuda.is_available(), "A CUDA GPU is required for this test."

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map="auto",
)

print("GPU:", torch.cuda.get_device_name(0))
print("Model dtype:", next(model.parameters()).dtype)
print("Model device map:", getattr(model, "hf_device_map", "not available"))

assert next(model.parameters()).dtype == torch.float16

config.json:   0%|          | 0.00/875 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/325 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

GPU: Tesla T4
Model dtype: torch.float16
Model device map: {'model.embed_tokens': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 0, 'model.layers.5': 0, 'model.layers.6': 0, 'model.layers.7': 0, 'model.layers.8': 0, 'model.layers.9': 0, 'model.layers.10': 0, 'model.layers.11': 0, 'model.layers.12': 0, 'model.layers.13': 0, 'model.layers.14': 1, 'model.layers.15': 1, 'model.layers.16': 1, 'model.layers.17': 1, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.layers.24': 1, 'model.layers.25': 1, 'model.layers.26': 1, 'model.layers.27': 1, 'model.layers.28': 1, 'model.layers.29': 1, 'model.layers.30': 1, 'model.layers.31': 1, 'model.norm': 1, 'model.rotary_emb': 1, 'lm_head': 1}


In [12]:
# Inspection.. Installed versins..
import torch
import peft
import transformers
import accelerate
import importlib.metadata as metadata

print("PyTorch:", torch.__version__)
print("PEFT:", peft.__version__)
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)

try:
    print("TorchAO:", metadata.version("torchao"))
except metadata.PackageNotFoundError:
    print("TorchAO: not installed")

PyTorch: 2.11.0+cu128
PEFT: 0.20.0
Transformers: 5.16.1
Accelerate: 1.14.0
TorchAO: 0.10.0


In [13]:
%pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
Note: you may need to restart the kernel to use updated packages.


In [14]:
# Verify Targets and Attach LoRA
from api.schemas.experiment import AdaptationConfig
from foundry.training.model_setup import attach_lora_adapters

adaptation = AdaptationConfig()

available_modules = {
    name.rsplit(".", 1)[-1]
    for name, _ in model.named_modules()
    if name
}

print("Configured targets:", adaptation.target_modules)
print(
    "Found targets:",
    sorted(set(adaptation.target_modules) & available_modules),
)

assert set(adaptation.target_modules).issubset(available_modules)

model = attach_lora_adapters(model, adaptation)
model.print_trainable_parameters()

trainable = [
    (name, parameter)
    for name, parameter in model.named_parameters()
    if parameter.requires_grad and "lora_" in name
]

assert trainable, "No trainable LoRA parameters were found."

print("Trainable LoRA tensors:", len(trainable))
print("LoRA attachment successful.")

Configured targets: ['q_proj', 'v_proj']
Found targets: ['q_proj', 'v_proj']
trainable params: 6,815,744 || all params: 8,037,076,992 || trainable%: 0.0848
Trainable LoRA tensors: 128
LoRA attachment successful.


In [16]:
# First RealWord Forward/Backward Pass

import torch

from foundry.datasets.record import DatasetRecord
from foundry.training.preprocessing import NAtlasPreprocessor
from foundry.training.collator import NAtlasDataCollator

# Small synthetic examples for pipeline validation only.
records = [
    DatasetRecord(
        id="lora-smoke-001",
        input="How are you?",
        target="Awa nɔ?",
        language_code="igl",
    ),
    DatasetRecord(
        id="lora-smoke-002",
        input="Please tell me how you are feeling today.",
        target="Awa nɔ?",
        language_code="igl",
    ),
]

preprocessor = NAtlasPreprocessor(
    tokenizer=tokenizer,
    max_length=512,
)

examples = [preprocessor.preprocess(record) for record in records]

collator = NAtlasDataCollator(tokenizer=tokenizer)
batch = collator(examples)

# Place inputs on the model's embedding device.
input_device = model.get_input_embeddings().weight.device
batch = {
    key: tensor.to(input_device)
    for key, tensor in batch.items()
}

model.train()
model.zero_grad(set_to_none=True)

outputs = model(**batch)
loss = outputs.loss

print("Loss:", loss.item())
print("Batch shape:", tuple(batch["input_ids"].shape))
print("Loss is finite:", torch.isfinite(loss).item())

assert loss is not None
assert torch.isfinite(loss).item(), "Training loss is not finite."

# Validate that gradients can flow into the LoRA parameters.
loss.backward()

gradient_parameters = [
    (name, parameter)
    for name, parameter in model.named_parameters()
    if "lora_" in name
    and parameter.requires_grad
    and parameter.grad is not None
]

assert gradient_parameters, "No LoRA parameters received gradients."

finite_gradients = [
    (name, parameter)
    for name, parameter in gradient_parameters
    if torch.isfinite(parameter.grad).all().item()
]

nonzero_gradients = [
    (name, parameter)
    for name, parameter in finite_gradients
    if torch.any(parameter.grad != 0).item()
]

print("LoRA parameters with gradients:", len(gradient_parameters))
print("LoRA parameters with finite gradients:", len(finite_gradients))
print("LoRA parameters with nonzero gradients:", len(nonzero_gradients))

assert finite_gradients, "All LoRA gradients are non-finite."
assert nonzero_gradients, "All LoRA gradients are zero."

print("\nSingle-batch forward/backward validation passed.")

Loss: 6.752442359924316
Batch shape: (2, 51)
Loss is finite: True
LoRA parameters with gradients: 128
LoRA parameters with finite gradients: 128
LoRA parameters with nonzero gradients: 64

Single-batch forward/backward validation passed.


In [ ]:
## LOCAL TESTs...  RUNNER TESTS..

In [6]:
# Focuses test
!pytest -q tests/test_natlas_training_runner.py

....                                                                     [100%]
4 passed in 12.99s


In [7]:
# Run related integration tests..
!pytest -q \
  tests/test_natlas_training_runner.py \
  tests/test_natlas_model_setup.py \
  tests/test_natlas_collator.py \
  tests/test_natlas_preprocessing.py

......................                                                   [100%]
22 passed in 3.68s
